In [207]:
import pandas as pd
import numpy as np

In [3]:
from pathlib import Path
import os

# remonte jusqu'au dossier qui contient .git, puis s'y place
RACINE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").is_dir())
os.chdir(RACINE)
print("racine projet :", RACINE)

racine projet : /Users/benjaminscemama/dev/market-risk-control


In [4]:
%load_ext sql
%config SqlMagic.autopandas = True
%sql duckdb:///:memory:
%sql ATTACH IF NOT EXISTS 'data/risk.db' AS r (TYPE sqlite);

Connecting to 'duckdb:///:memory:'

Running query in 'duckdb:///:memory:'

,Success


In [5]:
%%sql
-- Test configuration environnement
SELECT name FROM (SHOW ALL TABLES) WHERE database = 'r' ORDER BY name;

Running query in 'duckdb:///:memory:'

,name
0,mkt_forward_curve
1,mkt_spot_hourly
2,pos_snapshot
3,ref_contract
4,ref_customer
5,ref_site
6,trd_deal


In [6]:
fo = %sql select * from r.trd_deal;
fo_trie = fo.sort_values(["deal_id", "version"])
fo_derniere_version = fo_trie.drop_duplicates("deal_id", keep='last')
fo_en_vigueur = fo_derniere_version[fo_derniere_version["status"] == "CONFIRMED"]

Running query in 'duckdb:///:memory:'

In [7]:
fo["deal_id"].value_counts().value_counts()

count
1    8425
2     570
3       5
Name: count, dtype: int64

In [8]:
pred_toutes_lignes = pd.Series({
    "CONFIRMED": 0.90,
    "PENDING": 0.07,
    "CANCELLED": 0.03
})
pred_toutes_lignes

CONFIRMED    0.90
PENDING      0.07
CANCELLED    0.03
dtype: float64

In [9]:
pred_dernieres_versions = pd.Series({
    "CONFIRMED": 0.93,
    "PENDING": 0.038,
    "CANCELLED": 0.032
})
pred_dernieres_versions

CONFIRMED    0.930
PENDING      0.038
CANCELLED    0.032
dtype: float64

In [10]:
repartition_toutes_lignes = fo["status"].value_counts(normalize=True)
repartition_toutes_lignes

status
CONFIRMED    0.926722
PENDING      0.042902
CANCELLED    0.030376
Name: proportion, dtype: float64

In [11]:
repartition_dernieres_versions = fo_derniere_version["status"].value_counts(normalize=True)
repartition_dernieres_versions

status
CONFIRMED    0.926333
PENDING      0.042778
CANCELLED    0.030889
Name: proportion, dtype: float64

In [12]:
def erreur_relative(mesure, prediction):
    return (mesure - prediction).abs() / prediction

In [13]:
print(f"{erreur_relative(repartition_toutes_lignes, pred_toutes_lignes)} \n")
print(erreur_relative(repartition_dernieres_versions, pred_dernieres_versions))

status
CONFIRMED    0.029691
PENDING      0.387116
CANCELLED    0.012526
dtype: float64 

status
CONFIRMED    0.003943
PENDING      0.125731
CANCELLED    0.034722
dtype: float64


In [14]:
len(fo_derniere_version), fo_derniere_version["deal_id"].nunique()

(9000, 9000)

In [15]:
len(fo_en_vigueur)

8337

In [16]:
fo[fo.duplicated(keep=False)]["deal_id"].nunique()

40

In [17]:
fo["version"].value_counts()

version
1    9037
2     543
Name: count, dtype: int64

In [18]:
ids_trois_lignes = fo["deal_id"].value_counts()
ids_trois_lignes = ids_trois_lignes[ids_trois_lignes == 3].index
fo[fo["deal_id"].isin(ids_trois_lignes)].sort_values(["deal_id", "version"])

,deal_id,trade_date,trade_ts,commodity,direction,delivery_start,delivery_end,volume_mwh,price_eur_mwh,counterparty,book,status,version
313,D2600313,2025-06-05,2025-06-05 08:43:30,POWER,BUY,2026-06-01,2026-06-30,177.2,70.160,STATKRAFT,B2B_FR_POWER_HEDGE,CONFIRMED,1
9310,D2600313,2025-06-05,2025-06-06 08:43:30,POWER,BUY,2026-06-01,2026-06-30,185.7,68.693,STATKRAFT,B2B_FR_POWER_HEDGE,CONFIRMED,2
9556,D2600313,2025-06-05,2025-06-06 08:43:30,POWER,BUY,2026-06-01,2026-06-30,185.7,68.693,STATKRAFT,B2B_FR_POWER_HEDGE,CONFIRMED,2
1497,D2601497,2025-09-24,2025-09-24 11:53:37,POWER,SELL,2027-11-01,2027-11-30,265.3,86.853,UNIPER,B2B_FR_POWER_HEDGE,CONFIRMED,1
9545,D2601497,2025-09-24,2025-09-24 11:53:37,POWER,SELL,2027-11-01,2027-11-30,265.3,86.853,UNIPER,B2B_FR_POWER_HEDGE,CONFIRMED,1
9066,D2601497,2025-09-24,2025-09-25 11:53:37,POWER,SELL,2027-11-01,2027-11-30,240.6,87.908,UNIPER,B2B_FR_POWER_HEDGE,CONFIRMED,2
1680,D2601680,2025-09-22,2025-09-22 14:33:01,GAS,BUY,2027-02-01,2027-02-28,85.5,36.709,RWE,B2B_FR_STRUCT,CONFIRMED,1
9567,D2601680,2025-09-22,2025-09-22 14:33:01,GAS,BUY,2027-02-01,2027-02-28,85.5,36.709,RWE,B2B_FR_STRUCT,CONFIRMED,1
9001,D2601680,2025-09-22,2025-09-23 14:33:01,GAS,BUY,2027-02-01,2027-02-28,89.9,38.700,RWE,B2B_FR_STRUCT,CONFIRMED,2
1976,D2601976,2025-07-08,2025-07-08 17:34:22,GAS,BUY,2026-12-01,2026-12-31,216.2,38.439,ICE_ENDEX,B2B_FR_GAS_HEDGE,CONFIRMED,1


In [19]:
fo[fo["deal_id"].isin(ids_trois_lignes)].sort_values(["deal_id", "version"])["deal_id"].nunique()

5

In [20]:
fo[fo.duplicated(keep='first')]["version"].value_counts()

version
1    37
2     3
Name: count, dtype: int64

In [21]:
lignes_multi_versions = fo[fo["deal_id"].isin(
    fo["deal_id"].value_counts().loc[lambda s: s > 1].index
)].sort_values(["deal_id", "version"])

lignes_multi_versions["horodatage"] = pd.to_datetime(lignes_multi_versions["trade_ts"])
ecart_jours = lignes_multi_versions.groupby("deal_id")["horodatage"].agg(
    lambda s: (s.max() - s.min()).total_seconds() / 86400
)
ecart_jours.value_counts()

horodatage
1.0    540
0.0     35
Name: count, dtype: int64

In [22]:
fo_dates = fo.copy(deep=True)
fo_dates["trade_date"] = pd.to_datetime(fo_dates["trade_date"])
fo_dates["date_depuis_ts"] = pd.to_datetime(fo_dates["trade_ts"]).dt.normalize()

masque_dates_incoherentes = fo_dates["trade_date"] != fo_dates["date_depuis_ts"]

fo_dates_incoherentes = fo[masque_dates_incoherentes]

fo_dates_incoherentes

,deal_id,trade_date,trade_ts,commodity,direction,delivery_start,delivery_end,volume_mwh,price_eur_mwh,counterparty,book,status,version
9000,D2604989,2026-04-03,2026-04-04 16:24:55,GAS,SELL,2026-11-01,2027-01-31,211.8,35.225,EEX_CLEARED,B2B_FR_GAS_HEDGE,CONFIRMED,2
9001,D2601680,2025-09-22,2025-09-23 14:33:01,GAS,BUY,2027-02-01,2027-02-28,89.9,38.700,RWE,B2B_FR_STRUCT,CONFIRMED,2
9002,D2605511,2026-01-30,2026-01-31 10:40:45,POWER,BUY,2026-06-01,2026-06-30,333.7,49.700,VITOL,B2B_FR_POWER_HEDGE,CONFIRMED,2
9003,D2604854,2025-07-02,2025-07-03 14:53:34,GAS,SELL,2026-03-01,2026-03-31,661.9,31.564,VITOL,B2B_FR_GAS_HEDGE,CONFIRMED,2
9004,D2602053,2025-09-25,2025-09-26 09:56:15,POWER,SELL,2026-05-01,2026-05-31,557.0,73.451,ICE_ENDEX,B2B_FR_STRUCT,CONFIRMED,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9538,D2607408,2025-11-12,2025-11-13 10:01:58,POWER,BUY,2027-06-01,2028-05-31,398.0,71.325,TOTALENERGIES,B2B_FR_POWER_HEDGE,CONFIRMED,2
9539,D2602499,2025-12-01,2025-12-02 11:04:09,POWER,SELL,2026-03-01,2026-03-31,126.9,93.181,TOTALENERGIES,B2B_FR_POWER_HEDGE,CONFIRMED,2
9556,D2600313,2025-06-05,2025-06-06 08:43:30,POWER,BUY,2026-06-01,2026-06-30,185.7,68.693,STATKRAFT,B2B_FR_POWER_HEDGE,CONFIRMED,2
9564,D2601976,2025-07-08,2025-07-09 17:34:22,GAS,BUY,2026-12-01,2026-12-31,154.2,37.570,ICE_ENDEX,B2B_FR_GAS_HEDGE,CONFIRMED,2


In [23]:
!head -3 data/raw/bo_confirmations_20260724.csv

confirmation_id;deal_ref;trade_dt;product;buy_sell;del_from;del_to;quantity;unit;unit_price;ccy;cpty_code;state
BO905466;D2605829;23/10/2025;PWR_FR;B;01/12/2027;30/11/2028;255,500;MWH;91,9950;EUR;TOTALE;MATCHED
BO904587;D2604880;19/02/2026;PWR_FR;B;01/07/2026;31/07/2026;65,500;MWH;58,3570;EUR;EEX_CL;MATCHED


In [24]:
with open("data/raw/bo_confirmations_20260724.csv", "rb") as f:
    octets = f.read()

sorted({b for b in octets if b > 127})

[]

In [25]:
octets[:200]

b'confirmation_id;deal_ref;trade_dt;product;buy_sell;del_from;del_to;quantity;unit;unit_price;ccy;cpty_code;state\nBO905466;D2605829;23/10/2025;PWR_FR;B;01/12/2027;30/11/2028;255,500;MWH;91,9950;EUR;TOTA'

In [26]:
chemin_bo = "data/raw/bo_confirmations_20260724.csv"
bo = pd.read_csv(chemin_bo, sep=";", decimal=",",
                 parse_dates=["trade_dt", "del_from", "del_to"], dayfirst=True)

bo.dtypes

confirmation_id            object
deal_ref                   object
trade_dt           datetime64[ns]
product                    object
buy_sell                   object
del_from           datetime64[ns]
del_to             datetime64[ns]
quantity                  float64
unit                       object
unit_price                float64
ccy                        object
cpty_code                  object
state                      object
dtype: object

In [27]:
bo.head()

,confirmation_id,deal_ref,trade_dt,product,buy_sell,del_from,del_to,quantity,unit,unit_price,ccy,cpty_code,state
0,BO905466,D2605829,2025-10-23,PWR_FR,B,2027-12-01,2028-11-30,255.5,MWH,91.995,EUR,TOTALE,MATCHED
1,BO904587,D2604880,2026-02-19,PWR_FR,B,2026-07-01,2026-07-31,65.5,MWH,58.357,EUR,EEX_CL,MATCHED
2,BO907311,D2607778,2026-02-10,NG_PEG,B,2027-12-01,2028-02-29,463.9,MWH,34.725,EUR,VITOL,MATCHED
3,BO906063,D2606459,2026-03-10,NG_PEG,S,2026-05-01,2026-05-31,179.4,MWH,25.942,EUR,STATKR,MATCHED
4,BO904238,D2604505,2025-12-30,PWR_FR,B,2027-08-01,2027-08-31,153.7,MWH,72.358,EUR,TOTALE,MATCHED


In [28]:
bo.isna().sum()

confirmation_id    0
deal_ref           0
trade_dt           0
product            0
buy_sell           0
del_from           0
del_to             0
quantity           0
unit               0
unit_price         0
ccy                0
cpty_code          0
state              0
dtype: int64

In [29]:
bo_dates_us = pd.read_csv(chemin_bo, sep=";", decimal=",",
                          parse_dates=["trade_dt", "del_from", "del_to"])

for colonne in ["trade_dt", "del_from", "del_to"]:
    print(colonne, (bo_dates_us[colonne] != bo[colonne]).sum())

trade_dt 0
del_from 8249
del_to 0


/var/folders/49/r73y0kl571n0fg3wqtzp05c40000gn/T/ipykernel_76134/2827382643.py:1: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  bo_dates_us = pd.read_csv(chemin_bo, sep=";", decimal=",",
/var/folders/49/r73y0kl571n0fg3wqtzp05c40000gn/T/ipykernel_76134/2827382643.py:1: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  bo_dates_us = pd.read_csv(chemin_bo, sep=";", decimal=",",


In [30]:
len(bo), (bo["del_from"].dt.day == 1).mean(), bo["del_from"].dt.month.value_counts().sort_index()

(9010,
 np.float64(1.0),
 del_from
 1     761
 2     774
 3     744
 4     748
 5     741
 6     745
 7     737
 8     783
 9     736
 10    722
 11    780
 12    739
 Name: count, dtype: int64)

In [31]:
len(bo["deal_ref"]), bo["deal_ref"].nunique(), bo["deal_ref"].duplicated().sum()

(9010, 8945, np.int64(65))

In [32]:
ids_front = set(fo["deal_id"])
refs_bo = set(bo["deal_ref"])

len(refs_bo & ids_front), len(refs_bo - ids_front), len(ids_front - refs_bo)

(8665, 280, 335)

In [33]:
sorted(refs_bo - ids_front)[:20]

[' D2600031',
 ' D2600091',
 ' D2600492',
 ' D2600704',
 ' D2600803',
 ' D2600825',
 ' D2600964',
 ' D2601154',
 ' D2601155',
 ' D2601194',
 ' D2601262',
 ' D2601350',
 ' D2601539',
 ' D2601827',
 ' D2601905',
 ' D2601935',
 ' D2601961',
 ' D2602048',
 ' D2602106',
 ' D2602125']

In [34]:
orphelines_brutes = pd.Series(list(refs_bo - ids_front))

print(orphelines_brutes.str.len().value_counts().sort_index())
print()
print("canoniques telles quelles :", orphelines_brutes.str.match(r"^D\d{7}$").sum())
print("canoniques apres strip    :", orphelines_brutes.str.strip().str.match(r"^D\d{7}$").sum())

8    155
9    125
Name: count, dtype: int64

canoniques telles quelles : 95
canoniques apres strip    : 175


In [35]:
print("brut               :", orphelines_brutes.str.match(r"^D\d{7}$").sum())
print("strip              :", orphelines_brutes.str.strip().str.match(r"^D\d{7}$").sum())
print("strip + upper      :", orphelines_brutes.str.strip().str.upper().str.match(r"^D\d{7}$").sum())

brut               : 95
strip              : 175
strip + upper      : 235


In [36]:
orphelines_strip_upper = orphelines_brutes.str.strip().str.upper()

print("orphelines           :", len(orphelines_brutes))
print("forme canonique      :", orphelines_strip_upper.str.match(r"^D\d{7}$").sum())
print("existent au front    :", orphelines_strip_upper.isin(ids_front).sum())

orphelines           : 280
forme canonique      : 235
existent au front    : 140


In [37]:
ids_front_serie = pd.Series(sorted(ids_front))

print(len(ids_front_serie))
print(ids_front_serie.str.len().value_counts().sort_index())
print("canoniques :", ids_front_serie.str.match(r"^D\d{7}$").sum())

9000
8    9000
Name: count, dtype: int64
canoniques : 9000


In [38]:
orphelines_residuelles = orphelines_brutes[~orphelines_strip_upper.str.match(r"^D\d{7}$")]

print(orphelines_residuelles.str.len().value_counts().sort_index())
orphelines_residuelles.sample(20, random_state=0).tolist()

9    45
Name: count, dtype: int64


['D02602785',
 'D02607883',
 'D02608906',
 'D02605419',
 'D02603279',
 'D02603066',
 'D02607637',
 'D02605425',
 'D02605513',
 'D02603100',
 'D02605411',
 'D02601029',
 'D02603693',
 'D02606584',
 'D02606141',
 'D02604461',
 'D02605728',
 'D02602876',
 'D02603360',
 'D02607392']

In [39]:
orphelines_normalisees = (orphelines_brutes.str.strip()
                                            .str.upper()
                                            .str.replace(r"^D0+", "D", regex=True))

print("forme canonique   :", orphelines_normalisees.str.match(r"^D\d{7}$").sum())
print("existent au front :", orphelines_normalisees.isin(ids_front).sum())

forme canonique   : 280
existent au front : 185


In [40]:
refs_bo_serie = pd.Series(list(refs_bo))

refs_bo_serie.nunique(), (refs_bo_serie.str.strip()
                                       .str.upper()
                                       .str.replace(r"^D0+", "D", regex=True).nunique())

(8945, 8945)

In [41]:
bo["deal_key"] = (bo["deal_ref"].str.strip()
                                .str.upper()
                                .str.replace(r"^D0+", "D", regex=True))

bo["ref_corrigee"] = bo["deal_ref"] != bo["deal_key"]

In [42]:
print("lignes                       :", len(bo))
print("deal_key distinctes          :", bo["deal_key"].nunique())
print("lignes a reference sale      :", bo["ref_corrigee"].sum())
print("lignes pointant vers un deal :", bo["deal_key"].isin(ids_front).sum())

lignes                       : 9010
deal_key distinctes          : 8945
lignes a reference sale      : 187
lignes pointant vers un deal : 8915


In [43]:
lignes_orphelines = bo.loc[~bo["deal_key"].isin(ids_front)]

print("lignes    :", len(lignes_orphelines))
print("distinctes:", lignes_orphelines["deal_ref"].nunique())

lignes    : 95
distinctes: 95


In [44]:
cles_bo = set(bo["deal_key"])

ids_non_confirmes = ids_front - cles_bo
cles_orphelines   = cles_bo - ids_front

len(ids_non_confirmes), len(cles_orphelines), len(ids_front & cles_bo)

(150, 95, 8850)

In [45]:
jointure_naive = bo.merge(fo, left_on="deal_key", right_on="deal_id", how="inner")
len(jointure_naive)

9495

In [46]:
n_fo = fo["deal_id"].value_counts()
n_bo = bo["deal_key"].value_counts()
produit = n_bo.mul(n_fo, fill_value=0) 
produit.sum(), len(jointure_naive)

(np.float64(9495.0), 9495)

In [47]:
cles_appariees = sorted(ids_front & cles_bo)

a = n_bo[n_bo.index.isin(cles_appariees)] - 1
b = n_fo[n_fo.index.isin(cles_appariees)] - 1

a.sum(), b.sum(), (a * b).sum(), a.sum() + b.sum() + (a * b).sum()

(np.int64(65), np.int64(576), np.int64(4), np.int64(645))

In [48]:
croises = (a * b) > 0

print("cles concernees :", croises.sum())
pd.DataFrame({"a": a[croises], "b": b[croises], "produit": (a * b)[croises]})

cles concernees : 4


,a,b,produit
D2601918,1,1,1
D2604606,1,1,1
D2607974,1,1,1
D2608273,1,1,1


In [49]:
volume_naif = jointure_naive["volume_mwh"].sum()

fo_appariee = fo_derniere_version[fo_derniere_version["deal_id"].isin(cles_bo)]
volume_correct = fo_appariee["volume_mwh"].sum()

volume_naif, volume_correct, volume_naif - volume_correct, volume_naif / volume_correct - 1

(np.float64(2987513.8),
 np.float64(2790477.5999999996),
 np.float64(197036.2000000002),
 np.float64(0.07061020665423023))

In [50]:
valeur_naive = (jointure_naive["volume_mwh"] * jointure_naive["price_eur_mwh"]).sum()
valeur_correcte = (fo_appariee["volume_mwh"] * fo_appariee["price_eur_mwh"]).sum()

valeur_naive, valeur_correcte, valeur_naive - valeur_correcte, valeur_naive / valeur_correcte - 1

(np.float64(177141863.70790002),
 np.float64(165672694.967),
 np.float64(11469168.74090001),
 np.float64(0.06922787574128941))

In [51]:
fo["counterparty"].value_counts(), bo["cpty_code"].value_counts()

(counterparty
 RWE              830
 VITOL            820
 UNIPER           815
 EDF_TRADING      812
 STATKRAFT        807
 TOTALENERGIES    806
 GUNVOR           802
 MERCURIA         797
 SHELL_ENERGY     776
 AXPO             776
 ICE_ENDEX        770
 EEX_CLEARED      769
 Name: count, dtype: int64,
 cpty_code
 VITOL     770
 EDF_TR    770
 RWE       767
 STATKR    763
 MERCUR    756
 TOTALE    754
 UNIPER    752
 GUNVOR    749
 AXPO      736
 SHELL_    734
 EEX_CL    730
 ICE_EN    729
 Name: count, dtype: int64)

In [52]:
mapping = fo["counterparty"].drop_duplicates().to_frame()
mapping["code_bo"] = mapping["counterparty"].str[:6]

print("contreparties front :", len(mapping))
print("codes tronques      :", mapping["code_bo"].nunique())
print("codes bo non couverts :", set(bo["cpty_code"]) - set(mapping["code_bo"]))

contreparties front : 12
codes tronques      : 12
codes bo non couverts : set()


In [53]:
rec = bo.merge(fo_derniere_version, left_on="deal_key", right_on="deal_id", how="inner")

rec["cpty_ok"] = rec["cpty_code"] == rec["counterparty"].str[:6]
len(rec), rec["cpty_ok"].value_counts()

(8915,
 cpty_ok
 True    8915
 Name: count, dtype: int64)

In [54]:
for nom, table in [("toutes les lignes", fo),
                   ("premiere version", fo_trie.drop_duplicates("deal_id", keep="first")),
                   ("derniere version", fo_derniere_version),
                   ("en vigueur", fo_en_vigueur)]:
    print(f"{nom:20} {len(table):6} lignes   {table['volume_mwh'].sum():14,.1f} MWh")

toutes les lignes      9580 lignes      3,010,338.7 MWh
premiere version       9000 lignes      2,834,460.4 MWh
derniere version       9000 lignes      2,835,528.3 MWh
en vigueur             8337 lignes      2,616,247.1 MWh


In [61]:
paires = fo[fo["deal_id"].isin(
    fo["deal_id"].value_counts().loc[lambda s: s > 1].index)]

change = paires.groupby("deal_id").nunique()
change[[c for c in change.columns if c != "deal_id"]].gt(1).sum().sort_values(ascending=False)

trade_ts          540
volume_mwh        540
price_eur_mwh     540
version           540
trade_date          0
commodity           0
direction           0
delivery_start      0
delivery_end        0
counterparty        0
book                0
status              0
dtype: int64

In [57]:
p = paires.sort_values(["deal_id", "version"])
d = p.groupby("deal_id")[["volume_mwh", "price_eur_mwh"]].agg(["first", "last"])

dv = d[("volume_mwh", "last")] - d[("volume_mwh", "first")]
dp = d[("price_eur_mwh", "last")] - d[("price_eur_mwh", "first")]

pd.DataFrame({"delta_volume": dv.describe(), "delta_prix": dp.describe()})

,delta_volume,delta_prix
count,575.000000,575.000000
mean,1.857217,0.078824
std,63.375685,2.548494
min,-396.400000,-11.788000
25%,-15.950000,-1.313000
50%,0.000000,0.000000
75%,19.700000,1.290500
max,530.900000,9.861000


In [62]:
print(bo["unit"].value_counts(dropna=False))
print()
print(bo["quantity"].describe())

unit
MWH    8940
KWH      70
Name: count, dtype: int64

count    9.010000e+03
mean     2.639146e+03
std      3.371036e+04
min      9.700000e+00
25%      1.242250e+02
50%      2.224500e+02
75%      3.980000e+02
max      1.372200e+06
Name: quantity, dtype: float64


In [70]:
ordre = np.log10(bo["quantity"]).round()
pd.crosstab(ordre, bo["unit"])

unit,KWH,MWH
quantity,,
1.0,0,106
2.0,0,5837
3.0,0,2991
4.0,2,6
5.0,44,0
6.0,24,0


In [73]:
rec["ratio"] = rec["quantity"] / rec["volume_mwh"]
rec["ratio"].round(3).value_counts().head(10)

ratio
1.0       8845
1000.0      70
Name: count, dtype: int64

In [75]:
pd.crosstab(rec["ratio"].round(3), rec["unit"])

unit,KWH,MWH
ratio,,
1.0,0,8845
1000.0,70,0


In [78]:
mal_unite = np.isclose(rec["ratio"], 1000)

print("lignes          :", mal_unite.sum())
print("volume reel MWh :", rec.loc[mal_unite, "volume_mwh"].sum())

lignes          : 70
volume reel MWh : 20952.400000000005


In [79]:
pd.crosstab(rec["buy_sell"], rec["direction"])

direction,BUY,SELL
buy_sell,,
B,5139,24
S,31,3721


In [91]:
print("nb volume neg : ", (rec["volume_mwh"] < 0).sum())
print("nb quantity neg : ", (rec["quantity"] < 0).sum())

nb volume neg :  0
nb quantity neg :  0


In [85]:
rec.columns

Index(['confirmation_id', 'deal_ref', 'trade_dt', 'product', 'buy_sell',
       'del_from', 'del_to', 'quantity', 'unit', 'unit_price', 'ccy',
       'cpty_code', 'state', 'deal_key', 'ref_corrigee', 'deal_id',
       'trade_date', 'trade_ts', 'commodity', 'direction', 'delivery_start',
       'delivery_end', 'volume_mwh', 'price_eur_mwh', 'counterparty', 'book',
       'status', 'version', 'cpty_ok', 'ratio'],
      dtype='object')

In [172]:
discord = rec["buy_sell"].map({"B": "BUY", "S": "SELL"}) != rec["direction"]

tmp = rec.loc[discord, ["deal_id", "direction", "buy_sell", "volume_mwh", "quantity",
                  "price_eur_mwh", "unit_price", "commodity", "book", "state", "status"]]

tmp.head()

,deal_id,direction,buy_sell,volume_mwh,quantity,price_eur_mwh,unit_price,commodity,book,state,status
678,D2600502,BUY,S,141.8,141.8,27.409,27.409,GAS,B2B_FR_GAS_HEDGE,MATCHED,CONFIRMED
816,D2606793,BUY,S,406.5,406.5,38.510,38.510,GAS,B2B_FR_GAS_HEDGE,MATCHED,CONFIRMED
824,D2605558,BUY,S,135.1,135.1,73.807,73.807,POWER,B2B_FR_GAS_HEDGE,MATCHED,CONFIRMED
880,D2606086,BUY,S,154.8,154.8,71.149,71.149,POWER,B2B_FR_POWER_HEDGE,MATCHED,CONFIRMED
901,D2608596,SELL,B,328.3,328.3,95.059,95.059,POWER,B2B_FR_POWER_HEDGE,MATCHED,CONFIRMED


In [154]:
tmp[tmp["volume_mwh"] != tmp["quantity"]]

,deal_id,direction,buy_sell,volume_mwh,quantity,price_eur_mwh,unit_price,commodity,book
3989,D2608047,BUY,S,449.8,449800.0,94.818,94.82,POWER,B2B_FR_POWER_HEDGE


In [133]:
no_discord = rec["buy_sell"].map({"B" : "BUY", "S" : "SELL"}) == rec["direction"]
rec.loc[no_discord, ["deal_id", "direction", "buy_sell", "volume_mwh", "quantity",
                  "price_eur_mwh", "unit_price", "commodity", "book"]].head(15)

,deal_id,direction,buy_sell,volume_mwh,quantity,price_eur_mwh,unit_price,commodity,book
0,D2605829,BUY,B,255.5,255.5,91.995,91.995,POWER,B2B_FR_GAS_TRADING
1,D2604880,BUY,B,65.5,65.5,58.357,58.357,POWER,B2B_FR_STRUCT
2,D2607778,BUY,B,463.9,463.9,34.725,34.725,GAS,B2B_FR_POWER_HEDGE
3,D2606459,SELL,S,179.4,179.4,25.942,25.942,GAS,B2B_FR_GAS_HEDGE
4,D2604505,BUY,B,153.7,153.7,72.358,72.358,POWER,B2B_FR_POWER_HEDGE
5,D2600751,SELL,S,65.5,65.5,25.757,25.757,GAS,B2B_FR_GAS_HEDGE
6,D2607591,BUY,B,56.4,56.4,21.433,21.433,GAS,B2B_FR_POWER_HEDGE
7,D2603563,SELL,S,379.0,379.0,29.956,29.956,GAS,B2B_FR_GAS_HEDGE
8,D2601054,BUY,B,149.2,149.2,43.689,43.689,GAS,B2B_FR_GAS_HEDGE
9,D2601961,SELL,S,172.5,172.5,21.815,21.815,GAS,B2B_FR_GAS_HEDGE


In [156]:
pd.crosstab(rec.commodity, rec.book)

book,B2B_FR_GAS_HEDGE,B2B_FR_GAS_TRADING,B2B_FR_POWER_HEDGE,B2B_FR_POWER_TRADING,B2B_FR_STRUCT
commodity,,,,,
GAS,3183,72,100,101,91
POWER,119,134,4876,135,104


In [167]:
rec[["direction", "book"]].groupby("book").value_counts(normalize=True)

book                  direction
B2B_FR_GAS_HEDGE      BUY          0.583889
                      SELL         0.416111
B2B_FR_GAS_TRADING    SELL         0.572816
                      BUY          0.427184
B2B_FR_POWER_HEDGE    BUY          0.578577
                      SELL         0.421423
B2B_FR_POWER_TRADING  BUY          0.661017
                      SELL         0.338983
B2B_FR_STRUCT         BUY          0.610256
                      SELL         0.389744
Name: proportion, dtype: float64

In [169]:
tmp[["direction", "book"]].groupby("book").value_counts(normalize = True)

book                  direction
B2B_FR_GAS_HEDGE      SELL         0.545455
                      BUY          0.454545
B2B_FR_GAS_TRADING    BUY          0.500000
                      SELL         0.500000
B2B_FR_POWER_HEDGE    BUY          0.666667
                      SELL         0.333333
B2B_FR_POWER_TRADING  SELL         1.000000
Name: proportion, dtype: float64

In [165]:
rec["state"].value_counts(normalize=True)

state
MATCHED    0.969602
CXL        0.030398
Name: proportion, dtype: float64

In [166]:
tmp["state"].value_counts(normalize=True)

state
MATCHED    0.945455
CXL        0.054545
Name: proportion, dtype: float64

In [170]:
rec["status"].value_counts(normalize=True)

status
CONFIRMED    0.927089
PENDING      0.042513
CANCELLED    0.030398
Name: proportion, dtype: float64

In [173]:
tmp["status"].value_counts(normalize=True)

status
CONFIRMED    0.890909
PENDING      0.054545
CANCELLED    0.054545
Name: proportion, dtype: float64

In [186]:
pd.crosstab(rec.state, rec.status, normalize=True)

status,CANCELLED,CONFIRMED,PENDING
state,,,
CXL,0.030398,0.000000,0.000000
MATCHED,0.000000,0.927089,0.042513


In [190]:
len(tmp[tmp["status"] == "CONFIRMED"])

49

In [237]:
litiges      = rec[discord & (rec["status"] == "CONFIRMED")].copy()
concordantes = rec[(~discord) & (rec["status"] == "CONFIRMED")].copy()


def net_signe(table):
    return np.where(table["direction"] == "BUY",
                    table["volume_mwh"], -table["volume_mwh"]).sum()


net_concordantes = net_signe(concordantes)
net_litiges      = net_signe(litiges)

position_front = net_concordantes + net_litiges
position_bo    = net_concordantes - net_litiges
largeur        = abs(position_front - position_bo)

print(f"lignes litigieuses        : {len(litiges)}")
print(f"volume brut des litiges   : {litiges['volume_mwh'].sum():12,.1f} MWh")
print(f"net signe des litiges     : {net_litiges:12,.1f} MWh")
print()
print(f"position convention front : {position_front:12,.1f} MWh")
print(f"position convention bo    : {position_bo:12,.1f} MWh")
print(f"largeur indetermination   : {largeur:12,.1f} MWh")
print(f"part de la position       : {largeur / abs(position_front) * 100:12.3f} %")

lignes litigieuses        : 49
volume brut des litiges   :     14,409.4 MWh
net signe des litiges     :         42.8 MWh

position convention front :    435,523.7 MWh
position convention bo    :    435,438.1 MWh
largeur indetermination   :         85.6 MWh
part de la position       :        0.020 %
